# Hyperparameter tuning in AML

In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import dotenv
import os

dotenv.load_dotenv(override=True)

ml_client = ml_client = MLClient(
     credential         = DefaultAzureCredential()
    ,subscription_id    = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name= os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name     = os.environ["AZURE_ML"]
)


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## Defining the search space
This can be done with ```Choice``` in Azure ML.
- Defining a list of explicit values: ```Choice(values=[10,20,30])```
- Defining a range of values: ```Choice(values=range(1,10))```
- An arbitrary set of comma-separated values: ```Choice(values=(30,50,100))```

Discrete vlaues can also be selected from discrete distributions:
- ```QUniform(min_value, max_value, q)```: Returns a value like round(Uniform(min_value, max_value) / q) * q
- ```QLogUniform(min_value, max_value, q)```: Returns a value like round(exp(Uniform(min_value, max_value)) / q) * q
- ```QNormal(mu, sigma, q)```: Returns a value like round(Normal(mu, sigma) / q) * q
- ```QLogNormal(mu, sigma, q)```: Returns a value like round(exp(Normal(mu, sigma)) / q) * q

Continuous hyperparameters can also be defined as:
- ```Uniform(min_value, max_value)```: Returns a value uniformly distributed between min_value and max_value
- ```LogUniform(min_value, max_value)```: Returns a value drawn according to exp(Uniform(min_value, max_value)) so that the logarithm of the return value is uniformly distributed
- ```Normal(mu, sigma)```: Returns a real value that's normally distributed with mean mu and standard deviation sigma
- ```LogNormal(mu, sigma)```: Returns a value drawn according to exp(Normal(mu, sigma)) so that the logarithm of the return value is normally distributed

##### MLFlow

Watch out with MLFlow in Azure ML. ```mlflow.autolog()``` doesn't work because API changes after a certain version in MLFlow which AzureML relies on. 

In [ ]:
from azure.ai.ml import command
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

data_asset_name     = "diabetes-data-ml-table"
data_asset_version  = "0"
environment_name    = "diabetic_prediction_environment"

def create_base_job():
    job = command(
        code               = "./code"
        ,command            = "python train.py --data-asset ${{inputs.DATA}} " \
                            " -C ${{inputs.C}} --max-iter ${{inputs.max_iter}}"
        ,inputs             = {
                                "DATA": Input(type=AssetTypes.MLTABLE, path=f"azureml:{data_asset_name}:{data_asset_version}")
                                ,"C": 1
                                ,"max_iter": 100
                                }
        ,environment        = environment_name + "@latest"
        ,compute            = "diabetes-cluster"
        ,display_name       = "Diabetes hyperparameter tuning"
        ,experiment_name    = "train-diabetes-model-job"
    )

    return job 


**Create a GridSearch sweep job**

In [3]:
from azure.ai.ml.sweep import Choice, Normal

job = create_base_job()

command_job_for_sweep = job(
     C = Choice(values=[0.1, 1, 10])
    ,max_iter = Choice(values=[100, 500, 1000])
)

# Create a grid search sweep job
sweep_job = command_job_for_sweep.sweep(
     sampling_algorithm = "grid"
    ,primary_metric = "accuracy"
    ,goal="Maximize"
)

sweep_job.experiment_name = \
    "hyperparameter_tuning_grid_search_diabets_classifier"

ml_client.jobs.create_or_update(sweep_job)

Experiment,Name,Type,Status,Details Page
hyperparameter_tuning_grid_search_diabets_classifier,clever_school_5r44xh7vl2,sweep,Running,Link to Azure Machine Learning studio


**Random search**

In [5]:
from azure.ai.ml.sweep import Choice, LogUniform, QUniform, SweepJobLimits
import math

job = create_base_job()

command_job_for_sweep = job(
     C = LogUniform(math.log(1e-3), math.log(1e3))
    ,max_iter = QUniform(min_value=500, max_value=2000, q=1)
)

sweep_job_limits = SweepJobLimits(
     max_concurrent_trials = 4
    ,max_total_trials = 20
    ,timeout= 30*60           # In seconds
)

sweep_job = command_job_for_sweep.sweep(
     sampling_algorithm = "random"
    ,primary_metric = "accuracy"
    ,goal = "Maximize"
)

sweep_job.limits = sweep_job_limits
sweep_job.experiment_name = \
    "hyperparameter_tuning_random_search_diabets_classifier"

ml_client.jobs.create_or_update(sweep_job)

Experiment,Name,Type,Status,Details Page
hyperparameter_tuning_random_search_diabets_classifier,neat_rose_1gs24ml2z9,sweep,Running,Link to Azure Machine Learning studio


**Bayesian search (TPE)**

In [7]:
from azure.ai.ml.sweep import Choice, Uniform, QUniform, SweepJobLimits

job = create_base_job()

command_job_for_sweep = job(
     C = Uniform(10**-3, 10**3)     # Only unfiorm possible. You can handle this in the script by passing the exponents here and transform to log later. 
    ,max_iter = QUniform(min_value=500, max_value=2000, q=1)
)

sweep_job_limits = SweepJobLimits(
     max_concurrent_trials = 4
    ,max_total_trials = 20
    ,timeout= 30*60                 # In seconds
)

sweep_job = command_job_for_sweep.sweep(
     sampling_algorithm = "bayesian"
    ,primary_metric = "accuracy"
    ,goal = "Maximize"
)

sweep_job.limits = sweep_job_limits
sweep_job.display_name = "BayesianHyperParameterSearch"
sweep_job.experiment_name = \
    "hyperparameter_tuning_bayesian_search_diabets_classifier"

ml_client.jobs.create_or_update(sweep_job)

Experiment,Name,Type,Status,Details Page
hyperparameter_tuning_bayesian_search_diabets_classifier,shy_root_881r0h8l2b,sweep,Running,Link to Azure Machine Learning studio


**Random Sampling with Sobol**

In [ ]:
from azure.ai.ml.sweep import Choice, LogUniform, QUniform, SweepJobLimits, RandomSamplingAlgorithm
import math

job = create_base_job()

command_job_for_sweep = job(
     C = LogUniform(math.log(1e-3), math.log(1e3))
    ,max_iter = QUniform(min_value=500, max_value=2000, q=1)
)

sweep_job_limits = SweepJobLimits(
     max_concurrent_trials = 4
    ,max_total_trials = 20
    ,timeout= 30*60           # In seconds
)

sweep_job = command_job_for_sweep.sweep(
     sampling_algorithm = RandomSamplingAlgorithm(seed=123, rule="sobol")
    ,primary_metric = "accuracy"
    ,goal = "Maximize"
)

sweep_job.limits = sweep_job_limits
sweep_job.experiment_name = \
    "hyperparameter_tuning_random_search_diabets_classifier"

ml_client.jobs.create_or_update(sweep_job)

**Random Search with Sobol and Median stopping policy**

In [ ]:
from azure.ai.ml import command
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

data_asset_name     = "diabetes-data-ml-table"
data_asset_version  = "0"
environment_name    = "diabetic_prediction_environment"

def create_early_stopping_job():
    job = command(
        code               = "./code"
        ,command            = "python train_with_early_stopping.py --data-asset ${{inputs.DATA}} " \
                            " -alpha ${{inputs.alpha}} --max-iter ${{inputs.max_iter}}"
        ,inputs             = {
                                 "DATA": Input(type=AssetTypes.MLTABLE, path=f"azureml:{data_asset_name}:{data_asset_version}")
                                ,"alpha": 1
                                ,"max_iter": 100
                                }
        ,environment        = environment_name + "@latest"
        ,compute            = "diabetes-cluster"
        ,display_name       = "Diabetes hyperparameter tuning"
        ,experiment_name    = "train-diabetes-model-job-with-early-stopping"
    )

    return job 

In [ ]:
from azure.ai.ml.sweep import Choice, LogUniform, QUniform, \
                                SweepJobLimits, RandomSamplingAlgorithm, \
                                MedianStoppingPolicy, BanditPolicy, TruncationSelectionPolicy
import math

job = create_early_stopping_job()

command_job_for_sweep = job(
     alpha = LogUniform(math.log(1e-3), math.log(1e3))
    ,max_iter = QUniform(min_value=500, max_value=2000, q=1)
)

sweep_job_limits = SweepJobLimits(
     max_concurrent_trials = 4
    ,max_total_trials = 30
    ,timeout= 30*60           # In seconds
)

sweep_job = command_job_for_sweep.sweep(
     sampling_algorithm = RandomSamplingAlgorithm(seed=123, rule="sobol")
    ,primary_metric = "accuracy"
    ,goal = "Maximize"
)

sweep_job.limits = sweep_job_limits
sweep_job.experiment_name = \
    "hyperparameter_tuning_random_search_diabets_classifier"
sweep_job.early_termination = MedianStoppingPolicy(
                     delay_evaluation = 5               # Only start early stopping after x evaluations
                    ,evaluation_interval = 1            # Each time you log there needs to be an early stopping evaluation
                )

ml_client.jobs.create_or_update(sweep_job)

Uploading code (0.01 MBs): 100%|##########| 5880/5880 [00:00<00:00, 13106.81it/s]




Experiment,Name,Type,Status,Details Page
hyperparameter_tuning_random_search_diabets_classifier,affable_glove_w3hgmv005w,sweep,Running,Link to Azure Machine Learning studio
